In [1]:
import os
import sys

path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(
    splitted[0] + os.sep, splitted[1], splitted[2], splitted[3]
)
src_path = os.path.join(user_independent, "GitHub", "Photoswitching")
sys.path.append(src_path)

import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fcs_p
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.prediction as pr
import src.analysis as an
import src.transitions as tr

import numpy as np
import pandas as pd
from scipy.stats import expon
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

In [ ]:
fluorophores = fl.construct_fluorophores(
name="cy5_dna", distance=3, count=2, shape="square"
)
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

transitions = fluorophore_system.load_transitions(
summarize=True,
irradiance=5,
wavelength=640,
bleaching=True,
energy_transfer=True,
dstorm=True,
dstorm_parameters={'reducing_agent':'mea',
'concentration':100,
'ph':7.5},
energy_transfer_parameters={'overwrite': {'off': [1, 0.00001]}}
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.adjust_rates({8: 5e1})
transition_set.finalize()

transition_set.transition_df

transition_type  \
Fluorophore                       identity                                           
cy5_dna                           0                      TransitionType.EXCITATION   
                                  1            TransitionType.FLUORESCENT_EMISSION   
                                  2         TransitionType.INTERSYSTEM_CROSSING_ST   
                                  3                   TransitionType.ISOMERIZATION   
                                  4                      TransitionType.THERM_BISO   
                                  5                     TransitionType.REDUCTION_T   
                                  6                     TransitionType.REDUCTION_S   
                                  7                     TransitionType.OXIDATION_1   
                                  8                TransitionType.PHOTOBLEACHING_1   
                                  9               TransitionType.S1_S0_TRANSITIONS   
                                  10              TransitionType.T1_S0_TRANSITIONS   
                                  11             TransitionType.CIS_S0_TRANSITIONS   
D: cy5_dna, A: cy5_dna, dist: 3.0 12                     TransitionType.CIS_FRET_1   
                                  13                     TransitionType.CIS_FRET_2   
                                  14                     TransitionType.OFF_FRET_1   
                                  15                     TransitionType.OFF_FRET_2   
                                  16                           TransitionType.FRET   
                                  17               TransitionType.S_S_ANNIHILATION   
                                  18             TransitionType.S_T_ANNIHILATION_1   

                                           abbreviation        initial_state  \
Fluorophore                       identity                                     
cy5_dna                           0                 EXC       SingleState.S0   
                                  1                 FLU       SingleState.S1   
                                  2               ISCST       SingleState.S1   
                                  3                 ISO       SingleState.S1   
                                  4               TBISO      SingleState.Cis   
                                  5                REDT       SingleState.T1   
                                  6                REDS       SingleState.S1   
                                  7                OXI1     SingleState.OFF1   
                                  8                BLE1       SingleState.T1   
                                  9             S1S0SUM       SingleState.S1   
                                  10            T1S0SUM       SingleState.T1   
                                  11           CisS0SUM      SingleState.Cis   
D: cy5_dna, A: cy5_dna, dist: 3.0 12             CFRET1   PairedState.S1_Cis   
                                  13             CFRET2   PairedState.S1_Cis   
                                  14             OFRET1  PairedState.S1_OFF1   
                                  15             OFRET2  PairedState.S1_OFF1   
                                  16               FRET    PairedState.S1_S0   
                                  17                SSA    PairedState.S1_S1   
                                  18               STA1    PairedState.S1_T1   

                                                    final_state          rate  \
Fluorophore                       identity                                      
cy5_dna                           0              SingleState.S1  1.453925e+07   
                                  1              SingleState.S0  1.588235e+08   
                                  2              SingleState.T1  8.300000e+05   
                                  3             SingleState.Cis  4.000000e+06   
                                  4              SingleState.S0  5.000000e+03   
            

In [11]:
def style_dataframe(s):
    return [
        dict(
            selector="",
            props=[("background-color", "#081017"), ("color", "#898989"), ("border", "1px solid #db7921")]
        ),
        dict(
            selector="thead th",
            props=[("background-color", "#081017"), ("color", "#898989"), ("border", "1px solid #db7921")]
        ),
        dict(
            selector="tbody td",
            props=[("background-color", "#081017"), ("color", "#898989"), ("border", "1px solid #db7921")]
        )
    ]

transition_set.transition_df.style.set_table_styles(style_dataframe(transition_set.transition_df))